# Sample Downloader Pipeline

This notebook reads `sampled_spotify_tracks.csv`, downloads tracks with `deemix`, and writes a CSV with local file paths.

In [1]:
from pathlib import Path
import pandas as pd

from src.data.deemix_pipeline import download_tracks_from_csv

In [2]:
PROJECT_ROOT = Path('.').resolve()
INPUT_CSV = PROJECT_ROOT / 'sampled_spotify_tracks.csv'
OUTPUT_CSV = PROJECT_ROOT / 'sampled_spotify_tracks_with_paths.csv'
DOWNLOAD_DIR = PROJECT_ROOT / 'data' / 'raw' / 'deemix_downloads'

# Update this if your local deemix binary or output flag differs.
DEEMIX_BINARY = 'python -m deemix'
DEEMIX_OUTPUT_FLAG = '--path'
EXTRA_ARGS = []

INPUT_CSV, OUTPUT_CSV, DOWNLOAD_DIR

(PosixPath('/Volumes/MZ Music\uf028/genre-classifier/sampled_spotify_tracks.csv'),
 PosixPath('/Volumes/MZ Music\uf028/genre-classifier/sampled_spotify_tracks_with_paths.csv'),
 PosixPath('/Volumes/MZ Music\uf028/genre-classifier/data/raw/deemix_downloads'))

In [3]:
preview = pd.read_csv(INPUT_CSV).head(5)
preview[['track_id', 'artists', 'track_name']]

,track_id,artists,track_name
0,5nIrdpo8eXQU3YgZelEXkd,Vigiland;Tham Sway,Shots & Squats
1,55kDJdv7pymmG4URJlVTYR,Oliver Koletzki,Tankwa Town
2,6HfKUHjHuev7pYC0bFS1v5,Modeselektor,Tacken
3,2RoTe14GxtkU0ZzCgKys5X,Multani,Kalyan
4,2pSsHnjAgEPjHmet7ChlHQ,James Blake;SZA,Coming Back (feat. SZA)


## 1) Dry run (recommended first)

Creates the output CSV and commands without downloading files.

In [4]:
dry_run_df = download_tracks_from_csv(
    INPUT_CSV,
    csv_out=OUTPUT_CSV,
    output_dir=DOWNLOAD_DIR,
    deemix_binary=DEEMIX_BINARY,
    output_flag=DEEMIX_OUTPUT_FLAG,
    extra_args=EXTRA_ARGS,
    dry_run=True,
    limit=5,
)
dry_run_df[['track_id', 'deemix_query', 'deemix_status', 'download_filepath']]

[1/5] already_downloaded: https://open.spotify.com/track/5nIrdpo8eXQU3YgZelEXkd
[2/5] already_downloaded: https://open.spotify.com/track/55kDJdv7pymmG4URJlVTYR
[3/5] already_downloaded: https://open.spotify.com/track/6HfKUHjHuev7pYC0bFS1v5
[4/5] already_downloaded: https://open.spotify.com/track/2RoTe14GxtkU0ZzCgKys5X
[5/5] already_downloaded: https://open.spotify.com/track/2pSsHnjAgEPjHmet7ChlHQ


,track_id,deemix_query,deemix_status,download_filepath
0,5nIrdpo8eXQU3YgZelEXkd,https://open.spotify.com/track/5nIrdpo8eXQU3Yg...,already_downloaded,/Volumes/MZ Music/genre-classifier/data/raw/d...
1,55kDJdv7pymmG4URJlVTYR,https://open.spotify.com/track/55kDJdv7pymmG4U...,already_downloaded,/Volumes/MZ Music/genre-classifier/data/raw/d...
2,6HfKUHjHuev7pYC0bFS1v5,https://open.spotify.com/track/6HfKUHjHuev7pYC...,already_downloaded,/Volumes/MZ Music/genre-classifier/data/raw/d...
3,2RoTe14GxtkU0ZzCgKys5X,https://open.spotify.com/track/2RoTe14GxtkU0Zz...,already_downloaded,/Volumes/MZ Music/genre-classifier/data/raw/d...
4,2pSsHnjAgEPjHmet7ChlHQ,https://open.spotify.com/track/2pSsHnjAgEPjHme...,already_downloaded,/Volumes/MZ Music/genre-classifier/data/raw/d...


## 2) Real download run

Start with a small `limit` first (the CSV has 3600 rows).


In [5]:
import importlib
import src.data.deemix_pipeline as deemix_pipeline

importlib.reload(deemix_pipeline)
download_tracks_from_csv = deemix_pipeline.download_tracks_from_csv


In [6]:
download_df = download_tracks_from_csv(
    INPUT_CSV,
    csv_out=OUTPUT_CSV,
    output_dir=DOWNLOAD_DIR,
    deemix_binary=DEEMIX_BINARY,
    output_flag=DEEMIX_OUTPUT_FLAG,
    extra_args=EXTRA_ARGS,
    dry_run=False,
    timeout_sec=45,
    save_every=10,
)
download_df[['track_id', 'deemix_status', 'download_filepath']].head()

[1/3600] already_downloaded: https://open.spotify.com/track/5nIrdpo8eXQU3YgZelEXkd
[2/3600] already_downloaded: https://open.spotify.com/track/55kDJdv7pymmG4URJlVTYR
[3/3600] already_downloaded: https://open.spotify.com/track/6HfKUHjHuev7pYC0bFS1v5
[4/3600] already_downloaded: https://open.spotify.com/track/2RoTe14GxtkU0ZzCgKys5X
[5/3600] already_downloaded: https://open.spotify.com/track/2pSsHnjAgEPjHmet7ChlHQ
[6/3600] already_downloaded: https://open.spotify.com/track/7kXmJwrZGIhDaLT9sNo3ut
[7/3600] already_downloaded: https://open.spotify.com/track/5zSLlhe4kThCv0otCTv5CL
[8/3600] already_downloaded: https://open.spotify.com/track/5QRhs05R9MOXHQC2OOn5bq
[9/3600] already_downloaded: https://open.spotify.com/track/4kJWtxDDNb9oAk3h7sX3N4
[10/3600] already_downloaded: https://open.spotify.com/track/6re8Gb0JHD69k5B53tCAF6
[11/3600] already_downloaded: https://open.spotify.com/track/3x2djB2lM7twnO8D2M6A0C
[12/3600] already_downloaded: https://open.spotify.com/track/0RVmOh80HfpuygCBt2d1va
[

,track_id,deemix_status,download_filepath
0,5nIrdpo8eXQU3YgZelEXkd,already_downloaded,/Volumes/MZ Music/genre-classifier/data/raw/d...
1,55kDJdv7pymmG4URJlVTYR,already_downloaded,/Volumes/MZ Music/genre-classifier/data/raw/d...
2,6HfKUHjHuev7pYC0bFS1v5,already_downloaded,/Volumes/MZ Music/genre-classifier/data/raw/d...
3,2RoTe14GxtkU0ZzCgKys5X,already_downloaded,/Volumes/MZ Music/genre-classifier/data/raw/d...
4,2pSsHnjAgEPjHmet7ChlHQ,already_downloaded,/Volumes/MZ Music/genre-classifier/data/raw/d...


In [7]:
# Inspect results that failed or were not matched to a file path
issues = download_df[download_df['download_filepath'].fillna('') == '']
issues[['track_id', 'artists', 'track_name', 'deemix_status']].head(20)

,track_id,artists,track_name,deemix_status
24,2UkoP24VFjCccfeCo8b6v4,Ghauri;Noor Chahal,Yaariyan Lofi Flip,downloaded_no_file
80,28iXG2opEt5YUr1yZlcOfe,Steve Kroeger;Life of Kai,Summer,downloaded_no_file
83,2ggqqUQ5j358PFmvfBVswn,Ghauri;Lal Chand Yamla Jatt,Das main ki pyar wichon khatya (Sunno Flip),downloaded_no_file
104,2MZSXhq4XDJWu6coGoXX1V,Aphex Twin,Avril 14th,downloaded_no_file
142,4eaEUmyOmuIkDslaf7xw0f,DVBBS;Benny Benassi;Kyle Reynolds,Body Mind Soul (with Benny Benassi feat. Kyle ...,downloaded_no_file
187,47Lfe9HFfIxi0e8XMLlDKI,Tisoki;Shreya Jain;MAGIC;Shashwat Sachdev,Crazy Now,downloaded_no_file
284,6JrAEmBFd61eYIGVOSPnT4,Parra for Cuva;Anna Naklab,Wicked Games (feat. Anna Naklab) - Radio Edit,downloaded_no_file
313,1jy7SkRcmBCTcv4ZMtwz29,Jamie xx;Romy,Loud Places,downloaded_no_file
349,4PuvBMm0y4klls9NCg8lxl,HXE,Drop That MF,downloaded_no_file
364,0vZCG0H9KhtU7K8MEUVAoV,DVBBS;Borgeous,Tsunami,downloaded_no_file


In [9]:
download_df = download_tracks_from_csv(
    INPUT_CSV,
    csv_out=OUTPUT_CSV,
    output_dir=DOWNLOAD_DIR,
    deemix_binary="python -m deemix",
    output_flag="-p",
    dry_run=False,
    timeout_sec=45,
    limit=5,
    save_every=5,
)


[1/5] downloaded: https://open.spotify.com/track/5nIrdpo8eXQU3YgZelEXkd
[2/5] downloaded: https://open.spotify.com/track/55kDJdv7pymmG4URJlVTYR
[3/5] downloaded: https://open.spotify.com/track/6HfKUHjHuev7pYC0bFS1v5
[4/5] downloaded: https://open.spotify.com/track/2RoTe14GxtkU0ZzCgKys5X
[5/5] downloaded: https://open.spotify.com/track/2pSsHnjAgEPjHmet7ChlHQ
